# Sweep — Focal loss gamma

Proxy fold 0, TTA embeddings. 5 gamma values x 5 epochs. Winner: gamma=1.


In [1]:
from google.colab import drive
drive.mount("/content/drive")
%run colab_init.py

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=False)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
train.csv already at /content/train.csv
sample_submission.csv already at /content/sample_submission.csv
Baseline embeddings already staged at /content/embeddings_v2
TTA embeddings already staged at /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()
PROXY_FOLD = 0
EPOCHS = 5


In [4]:
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import BirdDataset, BirdMoE, FocalLoss, competition_macro_roc_auc, seed_everything
from birdclef.paths import SWEEPS_DIR
from birdclef.sweep_plots import plot_gamma_sweep

SAVE_DIR = SWEEPS_DIR / "gamma"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

GAMMA_VALUES = [0.5, 1.0, 2.0, 3.0, 5.0]
train_df = df_TTA[df_TTA["fold"] != PROXY_FOLD].reset_index(drop=True)
valid_df = df_TTA[df_TTA["fold"] == PROXY_FOLD].reset_index(drop=True)
train_loader = DataLoader(BirdDataset(train_df, is_tta=True), batch_size=64, shuffle=True, num_workers=0)
valid_loader = DataLoader(BirdDataset(valid_df, is_tta=True), batch_size=64, shuffle=False, num_workers=0)

sweep_results = {}
sweep_history = {}

for gamma in GAMMA_VALUES:
    print(f"Testing focal gamma={gamma}")
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = FocalLoss(gamma=gamma)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc = 0.0
    val_aucs = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if torch.isnan(inputs).any():
                continue
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        val_aucs.append(current_auc)
        best_auc = max(best_auc, current_auc)
        print(f"  Epoch {epoch + 1}/{EPOCHS} | Val AUC: {current_auc:.4f}")

    sweep_results[gamma] = best_auc
    sweep_history[gamma] = {"val_aucs": val_aucs}
    print(f"Peak AUC for gamma={gamma}: {best_auc:.4f}")

best_g = max(sweep_results, key=sweep_results.get)
print(f"Winner: gamma={best_g} (AUC={sweep_results[best_g]:.4f})")
plot_gamma_sweep(sweep_results, sweep_history, str(SAVE_DIR))


100%|██████████| 7110/7110 [00:09<00:00, 741.43it/s]


Testing focal gamma=0.5
  Epoch 1/5 | Val AUC: 0.9755
  Epoch 2/5 | Val AUC: 0.9751
  Epoch 3/5 | Val AUC: 0.9757
  Epoch 4/5 | Val AUC: 0.9770
  Epoch 5/5 | Val AUC: 0.9775
Peak AUC for gamma=0.5: 0.9775
Testing focal gamma=1.0
  Epoch 1/5 | Val AUC: 0.9754
  Epoch 2/5 | Val AUC: 0.9752
  Epoch 3/5 | Val AUC: 0.9762
  Epoch 4/5 | Val AUC: 0.9775
  Epoch 5/5 | Val AUC: 0.9778
Peak AUC for gamma=1.0: 0.9778
Testing focal gamma=2.0
  Epoch 1/5 | Val AUC: 0.9744
  Epoch 2/5 | Val AUC: 0.9749
  Epoch 3/5 | Val AUC: 0.9749
  Epoch 4/5 | Val AUC: 0.9769
  Epoch 5/5 | Val AUC: 0.9775
Peak AUC for gamma=2.0: 0.9775
Testing focal gamma=3.0
  Epoch 1/5 | Val AUC: 0.9745
  Epoch 2/5 | Val AUC: 0.9737
  Epoch 3/5 | Val AUC: 0.9750
  Epoch 4/5 | Val AUC: 0.9764
  Epoch 5/5 | Val AUC: 0.9762
Peak AUC for gamma=3.0: 0.9764
Testing focal gamma=5.0
  Epoch 1/5 | Val AUC: 0.9735
  Epoch 2/5 | Val AUC: 0.9724
  Epoch 3/5 | Val AUC: 0.9725
  Epoch 4/5 | Val AUC: 0.9741
  Epoch 5/5 | Val AUC: 0.9743
Peak A